# 🚀 المشروع النصفي لمقرر البيانات الضخمة - جامعة الرازي
## 🏗️ بناء خط بيانات هجين لمعالجة بيانات الطلبات (Hybrid ELT Data Pipeline)
**إعداد وتنفيذ:** مسار الطالب الفردي الكامل (10 / 10)  
**التقنيات المستخدمة:** `Python Streaming` | `Apache PySpark` | `MongoDB` | `ELT Architecture` | `Idempotency & Upsert`

---
### 📌 جدول مراحل التنفيذ وفق وثيقة التكليف (البنود 6.1 - 6.12):
0. **إعداد البيئة والمسارات**
1. **البند 6.1:** استخراج العينات الذكية بدون Pandas وبذاكرة $O(1)$.
2. **البند 6.2:** اختبار الموجه التلقائي للمحرك (File Router) بشرط الـ 200 MB.
3. **البند 6.6:** إعداد MongoDB وبناء واختبار الفهرس الفريد (Unique Index on `order_id`).
4. **البنود 6.7 و 6.8:** تجربة القواعد الثماني للتنظيف وسجل التدقيق (Audit Trail).
5. **البنود 6.3 و 6.4 و 6.5 و 6.9:** تشغيل خط البيانات الشامل (ELT Pipeline).
6. **البند 6.10:** إثبات الموثوقية وعدم التكرار (Idempotency Test).
7. **استعراض التكرار:** مقارنة السجلات المكررة في `orders_raw` قبل وبعد الدمج بـ `orders_validated`.
8. **معاينة السجلات بالتفصيل:** استعراض الصالحة النقية لوحدها والمصححة لوحدها والمعزولة لوحدها.
9. **البند 6.12:** التحقق من معادلة الاتساق والتقارير الفنية.
10. **محرك إصلاح العزل:** استرجاع السجلات المعزولة (Quarantine Recovery Engine).
11. **الاختبارات الآلية:** تشغيل حزمة الـ 15 Unit Tests.

### 0️⃣ إعداد المسارات واستيراد الحزم البرمجية

In [ ]:
print("=" * 75)
print("▶️  [المرحلة 0]: جاري استيراد الحزم والمكتبات وربط إعدادات المشروع وقراءة الثوابت...")
print("=" * 75)

import os
import sys
import json
from pathlib import Path
from pprint import pprint

# إضافة مسار المشروع إلى sys.path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# استيراد إعدادات المشروع ووحدات خط البيانات
from config.settings import (
    MONGO_URI,
    MONGO_DB_NAME,
    COLLECTION_RAW,
    COLLECTION_VALIDATED,
    COLLECTION_QUARANTINE,
    SMALL_FILE_THRESHOLD_MB,
    BATCH_SIZE,
    DATA_DIR,
    REPORTS_FILE_PATH,
)
from src.mongo_setup import get_mongo_client, get_database, setup_mongo_collections, reset_collections
from src.file_router import route_file
from src.quality_rules import process_and_classify_record, clean_arabic_numerals, clean_monetary_amount, clean_phone_number, clean_email, clean_date_format, clean_status_and_whitespace
from src.elt_pipeline import run_pipeline
from src.metrics import PipelineMetricsTracker
from src.show_duplicates_demo import analyze_and_show_duplicates
from src.reprocess_quarantine import run_quarantine_repair

print(f" - رابط قاعدة البيانات (MongoDB URI): {MONGO_URI}")
print(f" - اسم قاعدة البيانات (Database):     {MONGO_DB_NAME}")
print(f" - الحد الفاصل للموجه (Threshold):   {SMALL_FILE_THRESHOLD_MB} MB")
print(f" - حجم الدفعة الافتراضي (Batch Size): {BATCH_SIZE} سجل")

print("-" * 75)
print("✅ [اكتملت المرحلة 0 بنجاح]: تم استيراد كافة الحزم وربط الإعدادات بنجاح!")
print("-" * 75)

### 1️⃣ البند 6.1: استخراج عينة خفيفة بدون Pandas وبذاكرة $O(1)$

In [ ]:
print("=" * 75)
print("▶️  [البند 6.1]: جاري استخراج عينة اختبارية (50,000 سجل) بذاكرة O(1) دون تحميل كامل الملف في الذاكرة...")
print("=" * 75)

from src.create_small_sample import extract_sample_head

huge_csv_path = Path("H:/midterm-data-pipeline/data/orders_huge_mixed_quality.csv")
sample_csv_path = DATA_DIR / "orders_sample.csv"

if huge_csv_path.exists():
    sample_stats = extract_sample_head(huge_csv_path, sample_csv_path, num_rows=50000)
    pprint(sample_stats)
else:
    print(f"ℹ️ سيتم استخدام العينة الجاهزة مسبقاً في: {sample_csv_path}")

print("-" * 75)
print(f"✅ [اكتمل البند 6.1 بنجاح]: ملف العينة data/orders_sample.csv جاهز للاختبار السريع!")
print("-" * 75)

### 2️⃣ البند 6.2: اختبار الموجه التلقائي للمحرك (File Router)

In [ ]:
print("=" * 75)
print("▶️  [البند 6.2]: جاري فحص حجم الملف وتطبيق شرط الـ 200 MB لاختيار المحرك المناسب وطباعة التبرير الهندسـي...")
print("=" * 75)

# 1. فحص العينة الصغيرة
print("🔍 [فحص العينة الصغيرة]:")
engine_sample, meta_sample = route_file(str(sample_csv_path), threshold_mb=SMALL_FILE_THRESHOLD_MB)

# 2. فحص الملف الضخم إن وجد
if huge_csv_path.exists():
    print("\n🔍 [فحص الملف الضخم 12.65 GB]:")
    engine_huge, meta_huge = route_file(str(huge_csv_path), threshold_mb=SMALL_FILE_THRESHOLD_MB)

print("-" * 75)
print("✅ [اكتمل البند 6.2 بنجاح]: اتخذ الموجه التلقائي القرار الصحيح وطبع التبرير الهندسي وفق شرط الـ 200 MB!")
print("-" * 75)

### 3️⃣ البند 6.6: التحقق من اتصال MongoDB وإعداد واختبار الفهارس (Unique Index on `order_id`)

In [ ]:
print("=" * 75)
print("▶️  [البند 6.6]: جاري الاتصال بـ MongoDB وبناء واختبار الفهرس الفريد (Unique Index) على order_id...")
print("=" * 75)

from pymongo.errors import DuplicateKeyError

# 1. الاتصال بـ MongoDB وبناء فهارس الموثوقية
client = get_mongo_client()
db = get_database(client)
setup_info = setup_mongo_collections(db)

# 2. استعراض الفهارس المبنية والتأكد من وجود الفهرس الفريد UNIQUE
print("🔍 1. قائمة فهارس مجموعة orders_validated الحالية:")
for idx_name, idx_info in db[COLLECTION_VALIDATED].index_information().items():
    is_unique = " (🌟 UNIQUE فريد لمنع التكرار)" if idx_info.get("unique") else ""
    print(f"  - الفهرس: {idx_name:<22} -> الحقول: {idx_info['key']}{is_unique}")

# 3. اختبار عملي: محاولة إدخال سجلين مكررين عمداً بنفس order_id للتحقق من منعهما
print("\n🧪 2. اختبار عملي لمحاولة إدخال سجلين مكررين عمداً بنفس order_id:")
try:
    db[COLLECTION_VALIDATED].insert_one({"order_id": "TEST-DUP-999", "test": 1})
    print("  - تم إدخال السجل الأول (order_id: TEST-DUP-999) بنجاح.")
    
    # محاولة إدخال نفس رقم الطلب مرة ثانية لاختبار تصدي الفهرس له
    db[COLLECTION_VALIDATED].insert_one({"order_id": "TEST-DUP-999", "test": 2})
except DuplicateKeyError:
    print("  - 🛡️ تصدى MongoDB لمحاولة التكرار ورفض إدخال السجل الثاني بنجاح [DuplicateKeyError]!")
    print("  - ✅ النتيجة: الفهرس الفريد نشط ويعمل بنسبة 100% لمنع أي تكرار تجاري!")
finally:
    # تنظيف سجل الاختبار التجريبي
    db[COLLECTION_VALIDATED].delete_many({"order_id": "TEST-DUP-999"})

print("-" * 75)
print("✅ [اكتمل البند 6.6 بنجاح]: تم إثبات وجود وعمل الفهرس الفريد عملياً لضمان الـ Idempotency!")
print("-" * 75)

### 4️⃣ البنود 6.7 و 6.8: تجربة القواعد الثماني للتنظيف وسجل التدقيق (Audit Trail)

In [ ]:
print("=" * 75)
print("▶️  [البنود 6.7 و 6.8]: جاري اختبار القواعد الثماني للتنظيف وملاحظة توثيق سجل التدقيق corrections...")
print("=" * 75)

dirty_sample_record = {
    "order_id": "طلب-99901",
    "order_date": "31/01/2025",
    "status": "  مؤكد  ",
    "customer_id": "عميل-10",
    "customer_name": "محمد علي",
    "customer_phone": "+967 77 123 4567",
    "customer_email": "baker@@example..com",
    "city": "صنعاء",
    "district": "التحرير",
    "delivery_type": "سريع",
    "delivery_cost": "ألفان",
    "payment_method": "محفظة إلكترونية",
    "payment_status": "تم الدفع",
    "payment_amount": "٧٥٠٠٠٠٫٠",
    "currency": "ريال يمني",
    "total_amount": "750,000 YER",
    "items_json": '[{"sku":"SKU-101","name":"هاتف","qty":1,"unit_price":750000.0,"total":750000.0}]'
}

# معالجة وتصنيف السجل
result = process_and_classify_record(dirty_sample_record)

print(f"🔹 تصنيف السجل: [{result['classification']}]")
print(f"🔹 حالة الجودة:  [{result['record']['quality_status']}]")
print("\n🔹 سجل التدقيق (Audit Trail - Corrections):")
for c in result['record'].get('corrections', []):
    print(f"  - [{c['rule_code']}] الحقل: {c['field']} | القيمة الأصلية: '{c['original_value']}' -> المصححة: '{c['corrected_value']}'")

print("-" * 75)
print("✅ [اكتملت البنود 6.7 و 6.8 بنجاح]: تم تصحيح السجل وتوثيق أثر التعديل بدقة في كائن corrections!")
print("-" * 75)

### 5️⃣ البنود 6.3 و 6.4 و 6.5 و 6.9: تشغيل خط البيانات الشامل بالكامل (ELT Pipeline Execution)
ينفذ:
1. التحميل الخام (Raw Ingestion) إلى `orders_raw` دون تعديل مسبق.
2. معالجة وتدقيق السجلات داخل قاعدة البيانات.
3. عزل الأخطاء غير القابلة للإصلاح في `orders_quarantine`.
4. الكتابة الموثوقة بـ `Upsert` في `orders_validated`.

In [ ]:
print("=" * 75)
print("▶️  [البنود 6.3-6.9]: جاري تشغيل خط البيانات الشامل: التحميل الخام -> التدقيق والتنظيف -> العزل -> الكتابة بـ Upsert...")
print("=" * 75)

# تشغيل خط البيانات مع تصفير المجموعات لبدء تشغيل نظيف وموثق
report = run_pipeline(
    file_path=str(sample_csv_path),
    reset_db=True,
    batch_size=5000,
    threshold_mb=SMALL_FILE_THRESHOLD_MB
)

print("-" * 75)
print("✅ [اكتمل تنفيذ خط البيانات الشامل بنجاح!]")
print("-" * 75)

### 6️⃣ البند 6.10: إثبات الموثوقية وعدم التكرار (Idempotency Test)
إعادة تشغيل نفس الملف **دون تصفير المجموعات** للتأكد من أن `Inserted (New) = 0`.

In [ ]:
print("=" * 75)
print("▶️  [البند 6.10]: جاري إعادة تشغيل نفس البيانات لإثبات الـ Idempotency والتأكد من أن السجلات الجديدة المضافة = 0...")
print("=" * 75)

# إعادة تشغيل نفس الملف دون تصفير قاعدة البيانات
idempotency_report = run_pipeline(
    file_path=str(sample_csv_path),
    reset_db=False,
    batch_size=5000,
    threshold_mb=SMALL_FILE_THRESHOLD_MB
)

print("\n🎯 نتيجة اختبار الـ Idempotency:")
print(f" - السجلات الجديدة المضافة (Inserted New): {idempotency_report['upsert_metrics']['inserted_new']} (يجب أن تكون 0)")
print(f" - السجلات المحدثة (Updated Existing):     {idempotency_report['upsert_metrics']['updated_existing']}")
assert idempotency_report['upsert_metrics']['inserted_new'] == 0, "فشل اختبار التكرارية!"

print("-" * 75)
print("✅ [اكتمل البند 6.10 بنجاح]: نجح اختبار الموثوقية وتم إثبات عدم إنشاء أي سجل مكرر (Inserted New = 0)! 🎉")
print("-" * 75)

### 7️⃣ استعراض السجلات المكررة قبل الدمج وبعد الدمج (Duplicate Consolidation Demo)

In [ ]:
print("=" * 75)
print("▶️  [فحص التكرار]: جاري استعراض أمثلة واقعية لسجلات مكررة في orders_raw وكيف تم دمجها في وثيقة واحدة بـ orders_validated...")
print("=" * 75)

# تشغيل مستعرض التكرار والدمج
analyze_and_show_duplicates(limit_examples=1)

print("-" * 75)
print("✅ [اكتمل فحص التكرار بنجاح]: تم استعراض تكرارات البيانات الخام ودمجها بـ Upsert بنجاح!")
print("-" * 75)

### 8️⃣ معاينة تفصيلية للسجلات (الصالحة النقية 🟢 | المصححة آلياً 🟡 | المعزولة 🔴)

In [ ]:
print("=" * 75)
print("▶️  [معاينة السجلات المفصلة]: جاري استعلام السجلات الصالحة والمصححة والمعزولة من MongoDB...")
print("=" * 75)

# 1. السجلات الصالحة النقية (Pure Valid)
pure_valid = db[COLLECTION_VALIDATED].find_one({"quality_status": "valid"}, {"_id": 0})
pure_count = db[COLLECTION_VALIDATED].count_documents({"quality_status": "valid"})

print(f"🟢 1. البيانات الصالحة النقية (Pure Valid) - [العدد الإجمالي: {pure_count:,} سجل]:")
print("   (سجلات وصلت نظيفة 100% من المصدر بدون أي أخطاء)")
if pure_valid:
    print(f"   • رقم الطلب (order_id):        '{pure_valid.get('order_id')}'")
    print(f"   • اسم العميل:                  '{pure_valid.get('customer_name')}'")
    print(f"   • رقم الهاتف:                  '{pure_valid.get('customer_phone')}'")
    print(f"   • البريد الإلكتروني:           '{pure_valid.get('customer_email')}'")
    print(f"   • تاريخ الطلب (ISO):           '{pure_valid.get('order_date')}'")
    print(f"   • الإجمالي:                    {pure_valid.get('total_amount')} {pure_valid.get('currency')}")
    print(f"   • حالة الجودة (quality_status):'{pure_valid.get('quality_status')}'")
    print(f"   • التعديلات:                   {pure_valid.get('corrections', [])} (صفر تعديلات)")

# 2. السجلات المصححة آلياً (Auto-Corrected)
corrected = db[COLLECTION_VALIDATED].find_one({"quality_status": "corrected"}, {"_id": 0})
corr_count = db[COLLECTION_VALIDATED].count_documents({"quality_status": "corrected"})

print("\n" + "-" * 75)
print(f"🟡 2. البيانات المصححة آلياً (Auto-Corrected) - [العدد الإجمالي: {corr_count:,} سجل]:")
print("   (تم تصحيحها آلياً وتوثيق أثر التدقيق في مصفوفة corrections)")
if corrected:
    print(f"   • رقم الطلب (order_id):        '{corrected.get('order_id')}'")
    print(f"   • اسم العميل:                  '{corrected.get('customer_name')}'")
    print(f"   • الهاتف بعد التصحيح:          '{corrected.get('customer_phone')}'")
    print(f"   • الإيميل بعد التصحيح:         '{corrected.get('customer_email')}'")
    print(f"   • التاريخ الموحد (ISO):        '{corrected.get('order_date')}'")
    print(f"   • الإجمالي الرقمي:             {corrected.get('total_amount')} {corrected.get('currency')}")
    print(f"   • حالة الجودة (quality_status):'{corrected.get('quality_status')}'")
    print("   📋 سجل التدقيق الجنائي الموثق:")
    for idx, c in enumerate(corrected.get('corrections', []), 1):
        print(f"      ({idx}) [{c.get('rule_code')}] الحقل: '{c.get('field')}' | السابق: '{c.get('original_value')}' -> المصحح: '{c.get('corrected_value')}'")

# 3. السجلات المعزولة (Quarantined)
quar = db[COLLECTION_QUARANTINE].find_one({}, {"_id": 0})
quar_count = db[COLLECTION_QUARANTINE].count_documents({})

print("\n" + "-" * 75)
print(f"🔴 3. البيانات المعزولة (orders_quarantine) - [العدد الإجمالي: {quar_count:,} سجل]:")
print("   (سجلات تحتوي على أخطاء قاتلة يستحيل إصلاحها دون تخمين عشوائي)")
if quar:
    print(f"   • رقم الطلب (إن وجد):          '{quar.get('order_id')}'")
    print(f"   • كود الخطأ (Error Code):      '{quar.get('error_code')}'")
    print(f"   • سبب العزل التفصيلي:          '{quar.get('error_reason')}'")
    print(f"   • وقت وتاريخ العزل:            '{quar.get('quarantined_at')}'")

print("-" * 75)
print("✅ [اكتملت معاينة السجلات المفصلة بنجاح ووضوح تام دون أي اقتطاع!]")
print("-" * 75)

### 9️⃣ البند 6.12: التحقق من معادلة الاتساق والتقارير الفنية
$$\text{run\_raw\_count} = \text{run\_valid\_count} + \text{run\_corrected\_count} + \text{run\_quarantine\_count}$$

In [ ]:
print("=" * 75)
print("▶️  [البند 6.12]: جاري التحقق من معادلة الاتساق واستعراض ملخص التقرير الفني الموثق...")
print("=" * 75)

with open(REPORTS_FILE_PATH, "r", encoding="utf-8") as f:
    saved_report = json.load(f)

inv = saved_report["validation_invariants"]
print(f" - معادلة الاتساق مطبقة:    {inv['consistent']} ✅")
print(f" - إجمالي الخام المقروء:     {inv['raw_count']:,} سجل")
print(f" - إجمالي الصالح + المصحح:  {inv['processed_clean_total']:,} سجل")
print(f" - إجمالي المعزول:          {inv['quarantine_count']:,} سجل")
print(f" - الفرق الحسابي:           {inv['difference']} (صفر أخطاء)")
print(f" - سرعة الإدخال والتنظيف:   {saved_report['ingestion_metrics']['rows_per_second']:,.1f} سجل/ثانية")

print("-" * 75)
print("✅ [اكتمل البند 6.12 بنجاح]: تم إثبات معادلة الاتساق وتوثيق التقرير في reports/results.json!")
print("-" * 75)

### 🔟 تشغيل محرك إصلاح واسترجاع السجلات المعزولة (Quarantine Recovery Engine)

In [ ]:
print("=" * 75)
print("▶️  [محرك التعافي]: جاري تشغيل محرك الإصلاح الذكي لإغلاق JSON المقطوع وتصحيح التواريخ وترقية السجلات المعزولة...")
print("=" * 75)

# تشغيل محرك الإصلاح
repair_stats = run_quarantine_repair()

print("-" * 75)
print("✅ [اكتمل محرك التعافي بنجاح]: تم استرجاع وترقية السجلات المعزولة إلى orders_validated!")
print("-" * 75)

### 1️⃣1️⃣ تشغيل الاختبارات الآلية (Automated Unit Tests) داخل الدفتر

In [11]:
print("=" * 75)
print("▶️  [حزمة الاختبارات]: جاري تشغيل حزمة الاختبارات الآلية الـ 15 للتحقق من سلامة جميع القواعد والتصنيفات...")
print("=" * 75)

import unittest

suite = unittest.TestLoader().discover("tests")
runner = unittest.TextTestRunner(verbosity=2)
test_results = runner.run(suite)

print("-" * 75)
if test_results.wasSuccessful():
    print("🎉 [اكتملت جميع المراحل بنجاح]: جميع الاختبارات الـ 15 اجتازت بنجاح 100%! المشروع جاهز تماماً للمناقشة! 🏆")
print("-" * 75)

▶️  [حزمة الاختبارات]: جاري تشغيل حزمة الاختبارات الآلية الـ 15 للتحقق من سلامة جميع القواعد والتصنيفات...


test_corrected_record_audit_trail (test_classification.TestClassification)
A dirty record with Arabic numbers and messy email should be CORRECTED with audit trail. ... ok
test_quarantine_corrupted_json (test_classification.TestClassification)
Corrupted items_json ('not-json') must go to quarantine. ... ok
test_quarantine_empty_items (test_classification.TestClassification)
Empty items list must go to quarantine. ... ok
test_quarantine_impossible_date (test_classification.TestClassification)
Impossible year date must go to quarantine. ... ok
test_quarantine_missing_order_id (test_classification.TestClassification)
Missing order_id must go to quarantine. ... ok
test_recalculate_total_from_items_and_delivery (test_classification.TestClassification)
If total_amount is corrupted ('???') but items and delivery are valid, recalculate total. ... ok
test_valid_clean_record (test_classification.TestClassification)
A clean record should be classified as VALID without corrections. ... ok
test_rule

---------------------------------------------------------------------------
🎉 [اكتملت جميع المراحل بنجاح]: جميع الاختبارات الـ 15 اجتازت بنجاح 100%! المشروع جاهز تماماً للمناقشة! 🏆
---------------------------------------------------------------------------


ok

----------------------------------------------------------------------
Ran 15 tests in 0.028s

OK
